# Validación y Calidad de Datos

## Types validations
- using different kind ways

In [11]:
import re
from datetime import date

# VALIDACIÓN 1: TIPO DE DATO
def validar_tipo(valor, tipo_esperado):
    """Validar que el valor es del tipo esperado"""
    if tipo_esperado == "int":
        return isinstance(valor, int) and not isinstance(valor, bool)
    elif tipo_esperado == "float":
        return isinstance(valor, (int, float)) and not isinstance(valor, bool)
    elif tipo_esperado == "str":
        return isinstance(valor, str)
    elif tipo_esperado == "bool":
        return isinstance(valor, bool)
    elif tipo_esperado == "date":
        return isinstance(valor, date)
    return False

# VALIDACIÓN 2: RANGO
def validar_rango(valor, min_val, max_val):
    """Validar que valor está dentro de rango"""
    if valor is None:
        return True  # None es válido en algunos contextos
    return min_val <= valor <= max_val

# VALIDACIÓN 3: PATRÓN (regex)
def validar_patron(valor, patron):
    """Validar que valor cumple patrón"""
    if not isinstance(valor, str):
        return False
    return re.match(patron, valor) is not None

# VALIDACIÓN 4: CONJUNTO DE VALORES PERMITIDOS
def validar_enum(valor, valores_permitidos):
    """Validar que valor está en lista de permitidos"""
    return valor in valores_permitidos

# VALIDACIÓN 5: COMBINADA
def validar_registro(registro, esquema):
    """Validar un registro completo contra esquema"""
    errores = []
    
    for campo, reglas in esquema.items():
        valor = registro.get(campo)
        
        # Validar presencia (requerido)
        if reglas.get("requerido") and valor is None:
            errores.append(f"{campo}: faltante")
            continue
        
        # Validar tipo
        if valor is not None and "tipo" in reglas:
            if not validar_tipo(valor, reglas["tipo"]):
                errores.append(f"{campo}: tipo incorrecto (esperado {reglas['tipo']})")
                continue
        
        # Validar rango
        if valor is not None and "rango" in reglas:
            min_v, max_v = reglas["rango"]
            if not validar_rango(valor, min_v, max_v):
                errores.append(f"{campo}: fuera de rango [{min_v}, {max_v}]")
        
        # Validar patrón
        if valor is not None and "patron" in reglas:
            if not validar_patron(str(valor), reglas["patron"]):
                errores.append(f"{campo}: no cumple patrón {reglas['patron']}")
        
        # Validar enum
        if valor is not None and "enum" in reglas:
            if not validar_enum(valor, reglas["enum"]):
                errores.append(f"{campo}: valor no permitido (esperado {reglas['enum']})")
    
    return errores

# EJEMPLO DE USO
esquema_usuario = {
    "id": {"tipo": "int", "requerido": True},
    "nombre": {"tipo": "str", "requerido": True},
    "email": {"tipo": "str", "patron": r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"},
    "edad": {"tipo": "int", "rango": (0, 150)},
    "estado": {"enum": ["activo", "inactivo", "suspendido"]}
}

registros_prueba = [
    {"id": 1, "nombre": "Juan", "email": "juan@example.com", "edad": 30, "estado": "activo"},
    {"id": 2, "nombre": "María", "email": "invalid-email", "edad": 25, "estado": "activo"},
    {"id": None, "nombre": "Pedro", "email": "pedro@example.com", "edad": 999, "estado": "unknown"}
]

print("VALIDACIÓN DE REGISTROS:")
for reg in registros_prueba:
    errores = validar_registro(reg, esquema_usuario)
    if errores:
        print(f"  Registro {reg.get('id')}: ✗")
        for error in errores:
            print(f"    - {error}")
    else:
        print(f"  Registro {reg.get('id')}: ✓ válido")
      

VALIDACIÓN DE REGISTROS:
  Registro 1: ✓ válido
  Registro 2: ✗
    - email: no cumple patrón ^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$
  Registro None: ✗
    - id: faltante
    - edad: fuera de rango [0, 150]
    - estado: valor no permitido (esperado ['activo', 'inactivo', 'suspendido'])


## Reportes de Calidad de Datos
- Generar reportes automáticos sobre la calidad general.

In [12]:

import math

class ReporteCalidadDatos:
    """Generar reporte de calidad de datos"""
    
    def __init__(self, datos, esquema):
        self.datos = datos
        self.esquema = esquema
        self.metricas = {}
    
    def analizar(self):
        """Analizar calidad"""
        self.metricas = {
            "total_registros": len(self.datos),
            "campos": self._analizar_campos(),
            "completitud": self._completitud(),
            "validez": self._validez()
        }
        return self.metricas
    
    def _analizar_campos(self):
        """Analizar por campo"""
        campos = {}
        for campo in self.esquema.keys():
            valores = [r.get(campo) for r in self.datos]
            nulos = sum(1 for v in valores if v is None)
            campos[campo] = {
                "total": len(valores),
                "nulos": nulos,
                "completos": len(valores) - nulos,
                "pct_nulos": (nulos / len(valores) * 100) if valores else 0
            }
        return campos
    
    def _completitud(self):
        """Calcular completitud general"""
        total_celdas = len(self.datos) * len(self.esquema)
        celdas_nulas = sum(
            sum(1 for v in [r.get(c) for r in self.datos] if v is None)
            for c in self.esquema.keys()
        )
        celdas_completas = total_celdas - celdas_nulas
        pct_completitud = (celdas_completas / total_celdas * 100) if total_celdas > 0 else 0
        
        return {
            "celdas_totales": total_celdas,
            "celdas_nulas": celdas_nulas,
            "celdas_completas": celdas_completas,
            "porcentaje_completitud": pct_completitud
        }
    
    def _validez(self):
        """Contar registros válidos"""
        validos = 0
        invalidos = 0
        
        for registro in self.datos:
            # Validación simple: no tenga nulos en campos requeridos
            tiene_nulo = False
            for campo, reglas in self.esquema.items():
                if reglas.get("requerido") and registro.get(campo) is None:
                    tiene_nulo = True
                    break
            
            if tiene_nulo:
                invalidos += 1
            else:
                validos += 1
        
        return {
            "registros_validos": validos,
            "registros_invalidos": invalidos,
            "porcentaje_validez": (validos / len(self.datos) * 100) if self.datos else 0
        }
    
    def generar_reporte(self):
        """Imprimir reporte"""
        self.analizar()
        
        print("=" * 60)
        print("REPORTE DE CALIDAD DE DATOS")
        print("=" * 60)
        
        print(f"\nTotal de registros: {self.metricas['total_registros']}")
        
        print(f"\nCompletitud:")
        comp = self.metricas['completitud']
        print(f"  Celdas completas: {comp['celdas_completas']}/{comp['celdas_totales']}")
        print(f"  Porcentaje: {comp['porcentaje_completitud']:.1f}%")
        
        print(f"\nValidez:")
        val = self.metricas['validez']
        print(f"  Registros válidos: {val['registros_validos']}")
        print(f"  Registros inválidos: {val['registros_invalidos']}")
        print(f"  Porcentaje: {val['porcentaje_validez']:.1f}%")
        
        print(f"\nPor campo:")
        for campo, stats in self.metricas['campos'].items():
            print(f"  {campo}: {stats['completos']}/{stats['total']} ({100-stats['pct_nulos']:.0f}% completos)")
        
        # Calificar calidad general
        pct = self.metricas['completitud']['porcentaje_completitud']
        if pct >= 95:
            calificacion = "EXCELENTE"
        elif pct >= 85:
            calificacion = "BUENA"
        elif pct >= 75:
            calificacion = "ACEPTABLE"
        else:
            calificacion = "POBRE"
        
        print(f"\nCalificación general: {calificacion}")
        print("=" * 60)

# EJEMPLO
esquema = {
    "id": {"requerido": True},
    "nombre": {"requerido": True},
    "email": {"requerido": False},
    "edad": {"requerido": False}
}

datos = [
    {"id": 1, "nombre": "Juan", "email": "juan@example.com", "edad": 30},
    {"id": 2, "nombre": "María", "email": None, "edad": 25},
    {"id": None, "nombre": None, "email": "pedro@example.com", "edad": 35},
    {"id": 4, "nombre": "Ana", "email": None, "edad": None}
]

reporte = ReporteCalidadDatos(datos, esquema)
reporte.generar_reporte()
      

REPORTE DE CALIDAD DE DATOS

Total de registros: 4

Completitud:
  Celdas completas: 11/16
  Porcentaje: 68.8%

Validez:
  Registros válidos: 3
  Registros inválidos: 1
  Porcentaje: 75.0%

Por campo:
  id: 3/4 (75% completos)
  nombre: 3/4 (75% completos)
  email: 2/4 (50% completos)
  edad: 3/4 (75% completos)

Calificación general: POBRE


## Validación de Integridad Referencial
- Validar relaciones entre tablas (claves foráneas).

In [13]:

# VALIDACIÓN DE INTEGRIDAD REFERENCIAL
# Asegúrate que las claves foráneas existan

usuarios = [
    {"id": 1, "nombre": "Juan"},
    {"id": 2, "nombre": "María"},
    {"id": 3, "nombre": "Pedro"}
]

pedidos = [
    {"id": 101, "usuario_id": 1, "monto": 100},
    {"id": 102, "usuario_id": 2, "monto": 200},
    {"id": 103, "usuario_id": 999, "monto": 150},  # usuario_id no existe
    {"id": 104, "usuario_id": None, "monto": 50}   # usuario_id nulo
]

def validar_integridad_referencial(registros, campo_fk, valores_permitidos):
    """Validar que foreign key existe"""
    errores = []
    valores_ids = set(valores_permitidos)
    
    for reg in registros:
        fk_valor = reg.get(campo_fk)
        
        if fk_valor is None:
            errores.append({
                "registro": reg,
                "error": f"{campo_fk} es nulo"
            })
        elif fk_valor not in valores_ids:
            errores.append({
                "registro": reg,
                "error": f"{campo_fk}={fk_valor} no existe"
            })
    
    return errores

# VALIDAR
usuario_ids = [u["id"] for u in usuarios]
errores_integridad = validar_integridad_referencial(
    pedidos, 
    "usuario_id", 
    usuario_ids
)

print("VALIDACIÓN DE INTEGRIDAD REFERENCIAL:")
if errores_integridad:
    print(f"  Errores encontrados: {len(errores_integridad)}")
    for error in errores_integridad:
        print(f"    {error['registro']} -> {error['error']}")
else:
    print(f"  ✓ Todos los pedidos tienen usuario válido")

# ESTADÍSTICAS
print(f"\nResumen:")
print(f"  Pedidos válidos: {len(pedidos) - len(errores_integridad)}/{len(pedidos)}")
print(f"  Integridad: {(1 - len(errores_integridad)/len(pedidos))*100:.0f}%")
      

VALIDACIÓN DE INTEGRIDAD REFERENCIAL:
  Errores encontrados: 2
    {'id': 103, 'usuario_id': 999, 'monto': 150} -> usuario_id=999 no existe
    {'id': 104, 'usuario_id': None, 'monto': 50} -> usuario_id es nulo

Resumen:
  Pedidos válidos: 2/4
  Integridad: 50%


## Monitoreo Continuo de Calidad
- Trackear cambios de calidad a través del tiempo.

In [14]:
from datetime import date

class MonitorCalidad:
    """Monitorear cambios en calidad de datos"""
    
    def __init__(self):
        self.historial = []
    
    def registrar_snapshot(self, fecha, total_registros, completitud_pct, validez_pct):
        """Guardar snapshot de calidad"""
        self.historial.append({
            "fecha": fecha,
            "total_registros": total_registros,
            "completitud_pct": completitud_pct,
            "validez_pct": validez_pct
        })
    
    def generar_reporte_tendencia(self):
        """Ver tendencias de calidad"""
        if len(self.historial) < 2:
            return "No hay datos suficientes"
        
        print("=" * 60)
        print("TENDENCIA DE CALIDAD DE DATOS")
        print("=" * 60)
        
        primer = self.historial[0]
        ultimo = self.historial[-1]
        
        print(f"\nEvaluación:")
        print(f"  Primer snapshot: {primer['fecha']}")
        print(f"    Registros: {primer['total_registros']}")
        print(f"    Completitud: {primer['completitud_pct']:.1f}%")
        print(f"    Validez: {primer['validez_pct']:.1f}%")
        
        print(f"\n  Último snapshot: {ultimo['fecha']}")
        print(f"    Registros: {ultimo['total_registros']}")
        print(f"    Completitud: {ultimo['completitud_pct']:.1f}%")
        print(f"    Validez: {ultimo['validez_pct']:.1f}%")
        
        # Cambios
        cambio_comp = ultimo['completitud_pct'] - primer['completitud_pct']
        cambio_val = ultimo['validez_pct'] - primer['validez_pct']
        
        print(f"\nCambios:")
        print(f"  Completitud: {cambio_comp:+.1f}%")
        print(f"  Validez: {cambio_val:+.1f}%")
        
        # Alerta
        if cambio_comp < -5 or cambio_val < -5:
            print(f"\n⚠️ ALERTA: Calidad bajó significativamente")
        elif cambio_comp > 5 or cambio_val > 5:
            print(f"\n✓ Mejora: Calidad mejoró")
        else:
            print(f"\n→ Sin cambios significativos")
        
        print("=" * 60)

# EJEMPLO: Monitoreo a lo largo del tiempo
monitor = MonitorCalidad()

# Día 1
monitor.registrar_snapshot(date(2026, 2, 15), 1000, 85.0, 92.0)

# Día 8
monitor.registrar_snapshot(date(2026, 2, 22), 1200, 82.0, 88.0)

# Día 15
monitor.registrar_snapshot(date(2026, 3, 1), 1500, 80.0, 85.0)

# Generar reporte
monitor.generar_reporte_tendencia()
      

TENDENCIA DE CALIDAD DE DATOS

Evaluación:
  Primer snapshot: 2026-02-15
    Registros: 1000
    Completitud: 85.0%
    Validez: 92.0%

  Último snapshot: 2026-03-01
    Registros: 1500
    Completitud: 80.0%
    Validez: 85.0%

Cambios:
  Completitud: -5.0%
  Validez: -7.0%

⚠️ ALERTA: Calidad bajó significativamente


## Tips y Mejores Prácticas

- 💡 **Esquema único** — Defínelo UNA VEZ y reutilízalo para todos los validadores.
- ⚠️ **Validación defensiva** — Asume que todo puede ser malo hasta que se pruebe.
- 💡 **Reporta TODOS los errores** — No pares en el primero. Cliente necesita verlos todos.
- ℹ️ **Guarda historial** — Te ayuda a detectar degradación de calidad de datos.
- ⚠️ **No valides solo en interfaz** — El backend es donde es difícil burlarse.
- 💡 **Alertas automáticas** — Si completitud baja del 80%, notifica al equipo.

---

## Errores Comunes

### 1. No definir reglas de validación explícitamente

> **¿Por qué ocurre?**  
Si las reglas están implícitas en el código, nadie sabe qué se valida realmente.

> **Solución**  
Crea esquema explícito: `{"campo": {"requerido": true, "tipo": "int", "rango": [0, 100]}}`

---

### 2. Parar validación en el primer error

> **¿Por qué ocurre?**  
Cliente ve 1 error, lo arregla, vuelve a enviar, ahora hay otro. Ineficiente.

> **Solución**  
Recolecta TODOS los errores y devuelve lista completa.

---

### 3. Validar en el frontend solamente

> **¿Por qué ocurre?**  
El usuario puede inspeccionar el navegador y hacer bypass de la validación.

> **Solución**  
Valida SIEMPRE en backend, aunque también tengas validación en frontend.

---

### 4. No documentar por qué se rechazó un registro

> **¿Por qué ocurre?**  
Usuario no sabe qué está mal, no puede arreglar.

> **Solución**  
Mensajes claros: `"Email inválido (formato esperado: nombre@dominio.ext)"`

---

### 5. Usar regex mal y rechazar datos válidos

> **¿Por qué ocurre?**  
Email regex demasiado restrictiva rechaza `"nombre+etiqueta@example.com"`.

> **Solución**  
Usa regex simple que funciona para 99%: `r".+@.+ \..+"`